# Entrega 2 — Dataset y análisis exploratorio
**Predicción de la probabilidad de victoria en League of Legends antes de iniciar la partida**
Inteligencia Artificial 1 · Universidad Industrial de Santander

Este notebook acompaña el informe de la Entrega 2 y contiene el código con el que se obtuvieron sus cifras y gráficas.
Los datos provienen de la API oficial de Riot Games (región LAN, cola clasificatoria Solo/Dúo, 2026); se recolectaron con
el programa del repositorio `lol-win-predictor` y se exportaron a CSV con `python -m scripts.export_dataset`.

**Ejecución en Google Colab.** Suba los cinco archivos CSV de la entrega al panel de archivos de Colab (icono de carpeta
en la barra izquierda) y elija *Entorno de ejecución → Ejecutar todas*. Si los CSV no están ahí, el notebook los busca en
la carpeta `lol-win-predictor-data` de Google Drive y pide permiso para acceder a ella.

Fuera de Colab, los CSV deben estar en la misma carpeta del notebook o, dentro del repositorio, en `data/export/`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DRIVE_FOLDER = 'lol-win-predictor-data'   # carpeta en "Mi unidad" de Google Drive

# Los CSV se buscan primero junto al notebook (subidos a Colab o descomprimidos de la entrega),
# luego en Google Drive si se ejecuta en Colab y, por último, en data/export del repositorio.
IN_COLAB = 'google.colab' in sys.modules
if Path('matches.csv').exists():
    DATA_DIR = Path('.')
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive') / DRIVE_FOLDER
else:
    DATA_DIR = Path('..') / 'data' / 'export'

assert (DATA_DIR / 'matches.csv').exists(), f'No se encontró matches.csv en {DATA_DIR}'
pd.set_option('display.max_columns', 30)
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3})
GRIS, ACENTO = '#9aa0a6', '#1a73e8'
print('Leyendo datos de', DATA_DIR)

## 1. Carga de datos y tamaño

Los datos se reparten en cinco tablas: partidas, participantes, historial de cada jugador, jugadores semilla y rango
actual. Esta última es opcional y no se usa en el análisis. Los identificadores de los jugadores (`player_id`) están
**anonimizados** y no permiten recuperar la cuenta original de Riot.

In [ ]:
matches = pd.read_csv(DATA_DIR / 'matches.csv')
participants = pd.read_csv(DATA_DIR / 'participants.csv')
history = pd.read_csv(DATA_DIR / 'player_history.csv')
seeds = pd.read_csv(DATA_DIR / 'seed_players.csv')
snapshots = pd.read_csv(DATA_DIR / 'player_snapshots.csv')

tablas = {'matches': matches, 'participants': participants, 'player_history': history,
          'seed_players': seeds, 'player_snapshots': snapshots}
pd.DataFrame({'filas': {k: len(v) for k, v in tablas.items()},
              'columnas': {k: v.shape[1] for k, v in tablas.items()}})

In [ ]:
# Fechas legibles (la API entrega milisegundos desde 1970, en UTC)
matches['game_start'] = pd.to_datetime(matches['game_start_ts'], unit='ms', utc=True)
history['game_start'] = pd.to_datetime(history['game_start_ts'], unit='ms', utc=True)

valid = matches[matches['status'].isin(['fetched', 'history_done'])].copy()
print('Partidas descubiertas:', len(matches))
print('Partidas válidas:', len(valid), '| inválidas (remakes, línea faltante):', (matches['status'] == 'invalid').sum())
print('Partidas con historial completo de sus 10 jugadores:', (matches['status'] == 'history_done').sum())
print('Periodo de las partidas válidas:', valid['game_start'].min().date(), 'a', valid['game_start'].max().date())
print('Parches:', ', '.join(sorted(valid['patch'].astype(str).unique())))

## 2. Variable objetivo

`blue_win` vale 1 si gana el equipo azul (el que empieza en la esquina inferior izquierda del mapa) y 0 si gana el rojo.

In [ ]:
balance = valid['blue_win'].value_counts().sort_index()
tasa_azul = valid['blue_win'].mean()
ee = np.sqrt(tasa_azul * (1 - tasa_azul) / len(valid))
print(f'Victorias azul: {balance.get(1, 0)} | victorias rojo: {balance.get(0, 0)}')
print(f'Tasa de victoria azul: {tasa_azul:.1%} (IC 95 % aprox.: {tasa_azul - 1.96 * ee:.1%} a {tasa_azul + 1.96 * ee:.1%})')

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Rojo gana (0)', 'Azul gana (1)'], [balance.get(0, 0), balance.get(1, 0)], color=[GRIS, ACENTO])
ax.set_ylabel('Partidas')
ax.set_title(f'Clases casi balanceadas: el azul gana el {tasa_azul:.0%} de las partidas')
plt.tight_layout(); plt.show()

El equipo azul gana el 47,9 % de las veces y el intervalo de confianza incluye el 50 %, de modo que las clases están balanceadas. Como referencia mínima, predecir siempre la victoria del rojo acierta el 52,1 %.

## 3. Representatividad de la muestra

Se eligieron 40 jugadores por liga para que todas tuvieran el mismo peso. Cada jugador, sin embargo, aporta las partidas
que jugó en los 14 días anteriores, y en las ligas altas se juega con más frecuencia.

In [ ]:
ORDEN_LIGAS = ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND', 'MASTER', 'GRANDMASTER', 'CHALLENGER']
por_liga = (valid['seed_tier'].value_counts().reindex(ORDEN_LIGAS).fillna(0).astype(int))
semillas = seeds['tier'].value_counts().reindex(ORDEN_LIGAS).fillna(0).astype(int)
tabla_ligas = pd.DataFrame({'jugadores semilla': semillas, 'partidas válidas': por_liga,
                            '% de partidas': (por_liga / por_liga.sum() * 100).round(1)})
display(tabla_ligas)

apex = por_liga[['MASTER', 'GRANDMASTER', 'CHALLENGER']].sum() / por_liga.sum()
fig, ax = plt.subplots()
colores = [ACENTO if t in ('MASTER', 'GRANDMASTER', 'CHALLENGER') else GRIS for t in ORDEN_LIGAS]
ax.bar(ORDEN_LIGAS, por_liga.values, color=colores)
ax.axhline(por_liga.sum() / len(ORDEN_LIGAS), color='black', lw=1, ls='--', label='reparto igualitario')
ax.set_ylabel('Partidas válidas'); ax.legend()
ax.set_title(f'Master o superior aporta el {apex:.0%} de las partidas')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

Maestro, Gran Maestro y Retador aportan cerca de la mitad de las partidas, a pesar de agrupar a muy pocos jugadores. Es el sesgo más fuerte del conjunto de datos y se corrige en el preprocesamiento.

In [ ]:
fig, ax = plt.subplots()
por_dia = valid.set_index('game_start').resample('D').size()
ax.bar(por_dia.index, por_dia.values, color=GRIS, width=0.8)
ax.set_ylabel('Partidas'); ax.set_title('Partidas por día: ventana de 14 días antes de la recolección')
fig.autofmt_xdate(); plt.tight_layout(); plt.show()
print(valid['patch'].value_counts())

## 4. Participantes: posiciones y campeones

In [ ]:
print('Valores faltantes por columna (participants):')
print(participants.isna().sum().to_string())
print()
print('Jugadores por línea (debe ser 2 por partida en cada línea):')
print(participants['team_position'].value_counts().to_string())
print()
print('Campeones distintos:', participants['champion_name'].nunique())

top = participants['champion_name'].value_counts().head(15).sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top.index, top.values, color=GRIS)
ax.set_xlabel('Apariciones'); ax.set_title('Los 15 campeones más jugados (cola larga de campeones poco frecuentes)')
plt.tight_layout(); plt.show()

## 5. Historial de los jugadores antes de cada partida

En las partidas de la prueba piloto (estado `history_done`), la tabla `player_history` guarda hasta 10 partidas
clasificatorias que cada jugador disputó **antes** de la partida objetivo. Si alguna fuera posterior, el modelo tendría
acceso a información del futuro, así que esa condición se comprueba primero.

In [ ]:
target_start = matches.set_index('match_id')['game_start_ts']
fuga = (history['game_start_ts'] >= history['target_match_id'].map(target_start)).sum()
print('Partidas de historial que empiezan en o después de la partida objetivo (fuga temporal):', fuga)
assert fuga == 0

done = matches[matches['status'] == 'history_done']
print('Partidas objetivo con historial:', len(done), '| filas de historial:', len(history))
print()
print('Valores faltantes por columna (player_history):')
print(history.isna().sum().to_string())

In [ ]:
juegos_por_jugador = history.groupby(['target_match_id', 'player_id']).size()
esperados = len(done) * 10
con_historial = len(juegos_por_jugador)
print(f'Jugadores con al menos una partida previa: {con_historial} de {esperados}')
print(f'Jugadores con las 10 partidas completas: {(juegos_por_jugador == 10).sum()}')
print(f'Partidas previas que fueron remakes: {history["remake"].mean():.1%}')

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(juegos_por_jugador, bins=range(1, 12), color=GRIS, align='left', rwidth=0.85)
axes[0].set_xlabel('Partidas previas disponibles'); axes[0].set_ylabel('Jugadores')
axes[0].set_title('Historial disponible por jugador')
dias_antes = (history['target_match_id'].map(target_start) - history['game_start_ts']) / 864e5
axes[1].hist(dias_antes.clip(upper=60), bins=30, color=GRIS)
axes[1].set_xlabel('Días antes de la partida objetivo (recortado en 60)')
axes[1].set_title('Antigüedad de las partidas previas')
plt.tight_layout(); plt.show()

Casi todos los jugadores (981 de 990) tienen sus diez partidas previas, y la mayoría de esas partidas se jugó en los días inmediatamente anteriores.

### Valores atípicos en las partidas previas

In [ ]:
h = history[history['remake'] == 0].copy()
h['kda'] = (h['kills'] + h['assists']) / h['deaths'].clip(lower=1)
h['minutos'] = h['game_duration'] / 60
desc = h[['minutos', 'kills', 'deaths', 'assists', 'kda']].describe().T.round(2)
print(desc)

def atipicos_iqr(s):
    q1, q3 = s.quantile([0.25, 0.75])
    return ((s < q1 - 1.5 * (q3 - q1)) | (s > q3 + 1.5 * (q3 - q1))).mean()

print()
print('Proporción de atípicos (regla 1.5·IQR):')
print({c: f'{atipicos_iqr(h[c]):.1%}' for c in ['minutos', 'kills', 'deaths', 'assists', 'kda']})

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].boxplot([h['kills'], h['deaths'], h['assists']], tick_labels=['Asesinatos', 'Muertes', 'Asistencias'])
axes[0].set_title('KDA por partida: colas largas a la derecha')
axes[1].hist(h['minutos'], bins=30, color=GRIS)
axes[1].set_xlabel('Duración (minutos)'); axes[1].set_title('Duración de las partidas previas')
plt.tight_layout(); plt.show()

Los atípicos son partidas reales, con rendimientos excepcionales o duraciones muy largas, y no errores de registro. El KDA concentra la mayor proporción (cerca del 8 %); en el preprocesamiento se recortará en los percentiles 1 y 99 en lugar de eliminar filas.

## 6. Variables previas a la partida

A partir del historial se calculan, para cada jugador, las variables que plantea el proyecto. Las partidas previas
reiniciadas (*remakes*) se excluyen del cálculo.

| Variable | Qué mide |
|---|---|
| `winrate_10` | Proporción de victorias en sus partidas previas |
| `en_linea_habitual` | 1 si juega en la posición que más repite en su historial |
| `partidas_con_campeon` | Cuántas de sus partidas previas jugó con el mismo campeón |
| `racha` | Racha actual, positiva si viene ganando y negativa si viene perdiendo |
| `dias_desde_ultima` | Días transcurridos desde su partida anterior |
| `kda_medio` | KDA promedio en sus partidas previas |

Cada variable se promedia por equipo y se toma la diferencia **azul − rojo**, con lo que cada partida ocupa una sola fila.

In [ ]:
def racha_actual(wins):
    # wins ordenado de la más reciente a la más antigua
    if len(wins) == 0:
        return 0
    primero, n = wins[0], 0
    for w in wins:
        if w != primero:
            break
        n += 1
    return n if primero == 1 else -n

hh = history[history['remake'] == 0].sort_values(['target_match_id', 'player_id', 'history_rank'])
hh = hh.assign(kda=(hh['kills'] + hh['assists']) / hh['deaths'].clip(lower=1))
part = participants[participants['match_id'].isin(done['match_id'])]

filas = []
for (mid, pid), g in hh.groupby(['target_match_id', 'player_id']):
    filas.append({'match_id': mid, 'player_id': pid, 'partidas_previas': len(g),
                  'winrate_10': g['win'].mean(),
                  'linea_habitual': g['team_position'].replace('', np.nan).mode().iat[0] if g['team_position'].replace('', np.nan).notna().any() else np.nan,
                  'campeones_previos': list(g['champion_id']),
                  'racha': racha_actual(list(g['win'])),
                  'dias_desde_ultima': (target_start[mid] - g['game_start_ts'].iat[0]) / 864e5,
                  'kda_medio': g['kda'].mean()})
feats_jugador = part.merge(pd.DataFrame(filas), on=['match_id', 'player_id'], how='left')
feats_jugador['en_linea_habitual'] = (feats_jugador['team_position'] == feats_jugador['linea_habitual']).astype(float)
feats_jugador.loc[feats_jugador['linea_habitual'].isna(), 'en_linea_habitual'] = np.nan
feats_jugador['partidas_con_campeon'] = [
    np.nan if not isinstance(prev, list) else prev.count(c)
    for prev, c in zip(feats_jugador['campeones_previos'], feats_jugador['champion_id'])]

VARS = ['winrate_10', 'en_linea_habitual', 'partidas_con_campeon', 'racha', 'dias_desde_ultima', 'kda_medio']
print('Faltantes por variable (jugadores sin partidas previas):')
print(feats_jugador[VARS].isna().sum().to_string())
feats_jugador[VARS].describe().T.round(2)

In [ ]:
# Patrones descriptivos que se citan en el informe (sección 4)
fuera = feats_jugador['en_linea_habitual'] == 0
print(f'Jugadores fuera de su posición habitual: {fuera.mean():.0%}')
print('Por posición (%):')
print((fuera.groupby(feats_jugador['team_position']).mean() * 100).round(0).to_string())
distintos = hh.groupby(['target_match_id', 'player_id'])['champion_id'].nunique()
print(f'Campeones distintos en las diez partidas previas (mediana): {distintos.median():.0f}')
print(f'Jugadores con un campeón que no usaron en ninguna partida previa: {(feats_jugador["partidas_con_campeon"] == 0).mean():.0%}')
print(f'Jugadores que habían jugado otra partida menos de una hora antes: {(feats_jugador["dias_desde_ultima"] < 1 / 24).mean():.0%}')
print(f'Mayor pausa antes de la partida: {feats_jugador["dias_desde_ultima"].max():.0f} días')

Un tercio de los jugadores no está en su posición habitual, proporción que en la posición superior llega a cuatro de cada diez. También es frecuente elegir un campeón que no se jugó recientemente, y la mayoría de los jugadores encadena varias partidas en una misma sesión.

In [ ]:
equipo = feats_jugador.groupby(['match_id', 'team_id'])[VARS].mean().unstack('team_id')
dif = pd.DataFrame({v: equipo[(v, 100)] - equipo[(v, 200)] for v in VARS})
dif.columns = [f'dif_{v}' for v in VARS]
dataset = done[['match_id', 'blue_win', 'seed_tier', 'patch']].set_index('match_id').join(dif)
print('Filas del dataset de modelado (una por partida):', len(dataset))
dataset.head()

## 7. Relación de las variables con el resultado

### 7.1 A nivel de jugador

Con pocas partidas, la señal se aprecia mejor jugador por jugador. La pregunta es si quien llega en buena forma, en su
posición habitual o con un campeón conocido gana con más frecuencia. Cada partida aporta 10 observaciones que no son del
todo independientes, porque cinco de ellas comparten resultado, y por eso los intervalos son aproximados.

In [ ]:
fj = feats_jugador.dropna(subset=['winrate_10']).copy()
fj['forma'] = pd.cut(fj['winrate_10'], bins=[-0.01, 0.4, 0.6, 1.0], labels=['< 40 %', '40–60 %', '> 60 %'])
fj['linea'] = fj['en_linea_habitual'].map({1.0: 'Línea habitual', 0.0: 'Otra línea'})
fj['campeon'] = pd.cut(fj['partidas_con_campeon'], bins=[-1, 0, 2, 10], labels=['0 partidas', '1–2', '3 o más'])
fj['racha_g'] = pd.cut(fj['racha'], bins=[-11, -2, 1, 11], labels=['2+ derrotas', 'Sin racha', '2+ victorias'])

def tasa(col):
    t = fj.groupby(col, observed=False)['win'].agg(['mean', 'size'])
    t['ic95'] = 1.96 * np.sqrt(t['mean'] * (1 - t['mean']) / t['size'].clip(lower=1))
    return t

paneles = [('forma', 'Forma reciente (winrate previo)'), ('linea', 'Línea jugada'),
           ('campeon', 'Partidas previas con el campeón'), ('racha_g', 'Racha antes de la partida')]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
for ax, (col, titulo) in zip(axes, paneles):
    t = tasa(col)
    ax.bar(t.index.astype(str), t['mean'], yerr=t['ic95'], color=GRIS, capsize=3)
    for i, n in enumerate(t['size']):
        ax.text(i, 0.04, f'n={n}', ha='center', fontsize=8)
    ax.axhline(0.5, color='black', lw=1, ls='--'); ax.set_ylim(0, 1); ax.set_title(titulo, fontsize=10)
    ax.tick_params(axis='x', labelsize=8)
axes[0].set_ylabel('Tasa de victoria en la partida objetivo')
plt.suptitle(f'Victoria del jugador según su historial previo ({len(fj)} jugadores)'); plt.tight_layout(); plt.show()
for col, titulo in paneles:
    print(titulo); print(tasa(col).round(3).to_string()); print()
print('Correlación (punto-biserial) con la victoria del jugador:')
print(fj[VARS].corrwith(fj['win']).round(3).to_string())

En los cuatro paneles la tasa de victoria se mantiene cerca del 50 % y los intervalos cruzan esa línea. Ninguna de las cuatro variables separa por sí sola a los jugadores que ganan de los que pierden. El sistema de emparejamiento, que asigna rivales más fuertes a quien viene ganando, podría explicar parte de este resultado.

### 7.2 A nivel de partida

Se calcula la correlación de cada diferencia azul − rojo con `blue_win`. Con 99 partidas solo una correlación grande se
distingue del ruido, y las líneas punteadas marcan ese umbral aproximado (±1,96/√n).

In [ ]:
corr = dataset[[c for c in dataset.columns if c.startswith('dif_')]].corrwith(dataset['blue_win']).sort_values()
umbral = 1.96 / np.sqrt(dataset['blue_win'].notna().sum())
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(corr.index, corr.values, color=[ACENTO if abs(v) > umbral else GRIS for v in corr.values])
for x in (-umbral, umbral):
    ax.axvline(x, color='black', lw=1, ls='--')
ax.set_xlabel('Correlación con blue_win')
ax.set_title(f'Correlación con el resultado (línea punteada = ruido, n = {len(dataset)})')
plt.tight_layout(); plt.show()
print(corr.round(3).to_string())

Ninguna barra cruza el umbral. La correlación más alta, de −0,14, corresponde a los días desde la última partida, y con 99 partidas ninguna diferencia entre equipos anticipa al ganador.

In [ ]:
# Tasa de victoria azul según la ventaja en forma reciente (winrate_10)
d = dataset.dropna(subset=['dif_winrate_10']).copy()
d['grupo'] = pd.cut(d['dif_winrate_10'], bins=[-1, -0.05, 0.05, 1],
                    labels=['Rojo con mejor forma', 'Parejos', 'Azul con mejor forma'])
tabla = d.groupby('grupo', observed=False)['blue_win'].agg(['mean', 'size'])
print(tabla)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.bar(tabla.index.astype(str), tabla['mean'], color=[GRIS, GRIS, ACENTO])
ax.axhline(0.5, color='black', lw=1, ls='--')
ax.set_ylim(0, 1); ax.set_ylabel('Tasa de victoria azul')
ax.set_title('Victoria azul según la diferencia de forma reciente')
plt.tight_layout(); plt.show()

In [ ]:
# Correlación entre variables (a nivel de jugador): redundancia antes de modelar
m = feats_jugador[VARS].corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(m, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(VARS))); ax.set_xticklabels(VARS, rotation=45, ha='right')
ax.set_yticks(range(len(VARS))); ax.set_yticklabels(VARS)
for i in range(len(VARS)):
    for j in range(len(VARS)):
        ax.text(j, i, f'{m.iat[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Correlación entre variables de jugador'); plt.tight_layout(); plt.show()

Las variables se relacionan poco entre sí, con algunas excepciones esperables. El porcentaje de victorias se asocia con la racha (0,52) y con el KDA medio (0,41), y jugar en la posición habitual con repetir campeón (0,35). Ninguna pareja es tan alta como para que una variable resulte redundante, aunque habrá que tenerlo presente al interpretar los coeficientes de la regresión logística.

## 8. Resumen de la calidad de los datos

In [ ]:
resumen = pd.DataFrame([
    ('Partidas inválidas descartadas (remakes, línea faltante)', int((matches['status'] == 'invalid').sum())),
    ('Partidas válidas', len(valid)),
    ('Partidas con historial completo (prueba piloto)', len(done)),
    ('Jugadores sin ninguna partida previa', int(esperados - con_historial)),
    ('Partidas previas que fueron remake (excluidas de las variables)', int(history['remake'].sum())),
    ('Filas de historial con fuga temporal', int(fuga)),
    ('% de partidas de Master o superior', round(float(apex) * 100, 1)),
    ('Jugadores con rango actual consultado (snapshot)', len(snapshots)),
], columns=['Indicador', 'Valor'])
resumen

## 9. Plan de preprocesamiento

Corresponde al plan de la sección 5 del informe.

1. **Partidas no válidas.** Se conservan solo las partidas válidas, que el programa ya filtra, y se ignoran los *remakes* del historial.
2. **Desbalance por liga.** Se toma el mismo número de partidas por liga (`seed_tier`) o se pondera cada partida según su liga, para corregir el exceso de Maestro o superior.
3. **Historial incompleto.** Las variables se calculan con las partidas disponibles y se agrega el número de partidas previas (`partidas_previas`) como variable, para que el modelo sepa qué tan confiable es cada valor.
4. **Valores extremos.** Se recortan en los percentiles 1 y 99, y los días desde la última partida se transforman con un logaritmo. Las filas no se eliminan porque son partidas reales.
5. **Variables sin señal.** Se construyen variables relativas que comparan a cada jugador con su rival directo en la misma posición.

La posición se codificará con variables indicadoras (*one-hot*) y las variables numéricas se estandarizarán para la
regresión logística, paso que el bosque aleatorio no necesita. La partición entre entrenamiento y prueba será
cronológica, con las partidas más antiguas para entrenar y las más recientes para evaluar.